Our initial attempt for the regression tree

In [ ]:
set.seed(123)
merged <- read_csv("data/merged_data.csv")
split <- initial_split(merged, prop = 0.75)
train <- training(split)
test  <- testing(split)

library(rpart)

# 1. Drop non-numeric / ID columns
train_model <- train[, !names(train) %in% c("County")]

# 2. Sanitize column names — this is the key fix
names(train_model) <- make.names(names(train_model), unique = TRUE)

# 3. Fit the tree
fit <- rpart(outbreak ~ ., data = train_model, method = "anova")

library(rpart)

# 1. Drop non-numeric / ID columns
merged_model <- merged[, !names(merged) %in% c("County")]

# 2. Sanitize column names — this is the key fix
names(merged_model) <- make.names(names(merged_model), unique = TRUE)

# 3. Fit the tree
fit <- rpart(outbreak ~ ., data = merged_model, method = "anova", control = rpart.control(minsplit = 10, cp = 0.005))
fit

# WIP -----------------------------

# Load required libraries
library(tree)
library(ISLR)

# Set seed for reproducibility
set.seed(2)

# Remove the 'County' column FIRST, before sampling
merged_clean <- merged[, !names(merged) %in% c("County")]

#tree=tree(outbreak ~ . -County , merged ,subset=merged)


# other WIP-------------------------------



This section is for testing controls on the regression tree.

In [ ]:
# Make data input into predict
test_model <- test[, !names(test) %in% c("County")]
names(test_model) <- make.names(names(test_model), unique = TRUE)

optimal_tree <- rpart(
    formula = outbreak ~ .,
    data    = train_model,
    method  = "anova",
    control = list(minsplit = 7, maxdepth = 13, cp = 0.0100000)
    )
optimal_tree
pred <- predict(optimal_tree, newdata = test_model)
RMSE(pred = pred, obs = test_model$outbreak)
#pred

Unsuccessful SMOTE

In [ ]:

library(smotefamily)

set.seed(123)
merged <- read_csv("data/merged_data.csv")
split <- initial_split(merged, prop = 0.75)
train <- training(split)
test  <- testing(split)
train <- train[, !names(train) %in% c("County")]
train <- na.omit(train)
train_smote <- train[, colSums(is.na(train)) == 0]
non_num_cols <- names(train_smote)[!sapply(train_smote, is.numeric) & 
                                    names(train_smote) != "outbreak"]

sapply(train_smote[non_num_cols], function(x) any(x == "NA", na.rm = TRUE))

names(train_smote)[!sapply(train_smote, is.numeric)]

smote_output <- SMOTE(X = train_smote[, names(train_smote) != "outbreak"],
                      target = train_smote$outbreak)


# drop county col as value are str and cols with over half are NAs
train <- train[, !names(train) %in% c("county")]
train <- train[, colMeans(is.na(train)) < 0.1]
# This might be sketchy and 4 too
# ── 5. Impute remaining NAs with column medians ────────────────────────────
train <- train |>
  mutate(across(
    .cols = everything(),
    .fns  = ~ifelse(is.na(.), median(., na.rm = TRUE), .)
  ))

# Verify no NAs remain
stopifnot(sum(is.na(train)) == 0)

# ── 6. Ensure all feature columns are numeric ──────────────────────────────
non_num <- names(train)[!sapply(train, is.numeric) & names(train) != "outbreak"]
if (length(non_num) > 0) {
  train[non_num] <- lapply(train[non_num], as.numeric)
}

cat("Train rows:", nrow(train), "| Cols:", ncol(train), "\n")
cat("Class balance before SMOTE:\n")
print(table(train$outbreak))


In [ ]:
# use if we want tune on minsplit and maxdepth
hyper_grid <- expand.grid(
  minsplit = seq(5, 20, 1),
  maxdepth = seq(8, 15, 1)
)

models <- list()

for (i in 1:nrow(hyper_grid)) {

  # get minsplit, maxdepth values at row i
  minsplit <- hyper_grid$minsplit[i]
  maxdepth <- hyper_grid$maxdepth[i]

  # train a model and store in the list
  models[[i]] <- rpart(
    formula = outbreak ~ .,
    data    = train_balanced,
    method  = "class",
    control = list(minsplit = minsplit, maxdepth = maxdepth)
    )
}
get_cp <- function(x) {
  min    <- which.min(x$cptable[, "xerror"])
  cp <- x$cptable[min, "CP"] 
}

# function to get minimum error
get_min_error <- function(x) {
  min    <- which.min(x$cptable[, "xerror"])
  xerror <- x$cptable[min, "xerror"] 
}

hyper_grid %>%
  mutate(
    cp    = purrr::map_dbl(models, get_cp),
    error = purrr::map_dbl(models, get_min_error)
    ) %>%
  arrange(error) %>%
  top_n(-5, wt = error)

  # # 2. View the complexity parameter table
# printcp(dtree_model)

# # 3. Find the cp value with the minimum xerror
# opt_cp <- dtree_model$cptable[which.min(dtree_model$cptable[,"xerror"]), "CP"]

# # 4. Prune the tree using the optimized CP
# pruned_dtree_model <- prune(dtree_model, cp = opt_cp)

# pred <- predict(pruned_dtree_model, newdata = train, type = "class")

# dtree_table <- table(Actual = train$outbreak, Predicted = pred)
# dtree_table